# 📓 Notebook 02 — Network, Infra & Active Directory

Host discovery, service enum, SMB/NetBIOS, credential attacks, lateral movement, exploit delivery.

# 🧱 BASE (shared setup) — run these first, in order

In [ ]:
# Which GPU did you actually get? Decide the model from this.
!nvidia-smi
import torch, psutil
p = torch.cuda.get_device_properties(0)
print(f"GPU: {p.name} | VRAM {p.total_memory/1e9:.1f} GB | RAM {psutil.virtual_memory().total/1e9:.1f} GB")
!df -h /content | tail -1

In [ ]:
import os
try:
    if not os.path.exists('/content/drive/MyDrive'):
        from google.colab import drive
        drive.mount('/content/drive')
    CACHE = '/content/drive/MyDrive/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("cache:", CACHE)
except Exception as e:
    print("Drive notice:", e)
    CACHE = '/content/redteam'
    os.makedirs(CACHE, exist_ok=True)
    print("local cache:", CACHE)


In [ ]:
%%bash
apt-get -qq update
apt-get -qq install -y docker.io git curl wget jq unzip tmux nodejs npm python3-venv python3-pip net-tools iputils-ping dnsutils
# Colab blocks Docker's iptables hooks -> disable them
dockerd --iptables=false --ip6tables=false --bridge=none >/var/log/dockerd.log 2>&1 &
for i in $(seq 1 30); do docker info >/dev/null 2>&1 && break; sleep 2; done
docker info | grep -E "Server Version|Storage Driver"
echo "docker ready" 

In [ ]:
%%bash
curl -fsSL https://ollama.com/install.sh | sh || true
command -v ollama || { curl -L https://ollama.com/download/ollama-linux-amd64.tgz | tar -xz -C /usr; }
nohup ollama serve >/var/log/ollama.log 2>&1 &
for i in $(seq 1 40); do curl -s localhost:11434/api/version >/dev/null && break; sleep 2; done
curl -s localhost:11434/api/version && echo " <- ollama up" 

In [ ]:
%%bash
if ollama list | grep -q "qwen2.5-coder:7b"; then
  echo "qwen2.5-coder:7b ready"
else
  ollama pull qwen2.5-coder:7b
fi
ollama list


In [ ]:
%%bash
# HexStrike = 150+ tools behind one MCP endpoint
if [ ! -d /content/hexstrike ]; then
  git clone --depth 1 https://github.com/0x4m4/hexstrike-ai /content/hexstrike
fi
cd /content/hexstrike
if ! curl -sf localhost:8888/health >/dev/null; then
  nohup python3 hexstrike_server.py --port 8888 >/var/log/hexstrike.log 2>&1 &
  sleep 5
fi
echo "hexstrike:"; curl -s localhost:8888/health; echo

# Kali toolbox container the agent can shell into
docker pull -q kalilinux/kali-rolling || true
docker run -d --name redteam --network host kalilinux/kali-rolling sleep infinity || true
echo toolbox-ready


In [ ]:
%%bash
if ! pgrep -f "cloudflared tunnel --url http://localhost:8888" >/dev/null; then
  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared 2>/dev/null || true
  chmod +x /usr/local/bin/cloudflared 2>/dev/null || true
  nohup cloudflared tunnel --url http://localhost:8888 >/var/log/cf.log 2>&1 &
  sleep 10
fi
echo "HexStrike URL:"; grep -o 'https://[a-z0-9-]*\.trycloudflare\.com' /var/log/cf.log | head -1


In [ ]:
%%bash
apt-get -qq update && apt-get -qq install -y nmap masscan arp-scan hydra john hashcat smbmap nbtscan whois || true
pip -q install netexec impacket enum4linux-ng 2>/dev/null || true
[ ! -d /content/vulhub ] && git clone --depth 1 https://github.com/vulhub/vulhub /content/vulhub || true
echo "net tools ready"


In [ ]:
%%bash
# --- targets ---
docker run -d --name metasploitable2 --network host tleemcjr/metasploitable2
# SMB share host
docker run -d --name smb --network host dperson/samba -p -u "user;pass" -s "share;/tmp;yes;no;no;user"
# vulnerable FTP (vsftpd 2.3.4 backdoor)
docker run -d --name ftp --network host fauria/vsftpd
sleep 20
nmap -sV -T4 localhost -p 21,22,23,80,139,445,3306,5432,8080 | tail -30

In [ ]:
%%bash
# --- local exploit framework ---
docker exec redteam bash -lc 'msfconsole -q -x "version; exit"' | head -5
docker exec redteam bash -lc 'which nmap sqlmap hydra netexec responder' 2>/dev/null || true

In [ ]:
# --- example agent tasks ---
!nmap -sV -sC -p- -T4 localhost | tee /content/nmap_full.txt | tail -40
!enum4linux-ng -A localhost || true
!smbmap -H localhost -u user -p pass || true
!hydra -l user -P /usr/share/wordlists/rockyou.txt ssh://localhost -t 4 -f || true